# 03 - Chunking estructural del corpus normativo ambiental

Este notebook genera fragmentos estructurados (*chunks*) a partir de los textos extraídos de los PDFs.

Entrada principal:

- `data/metadata/corpus_normativo_ambiental_con_extraccion.csv`
- textos `.txt` ubicados en `data/processed/`

Salida generada:

- `data/chunks/chunks_normativa_v1.jsonl`
- `data/chunks/chunks_normativa_v1.csv`
- reportes en `experiments/resultados/`

El objetivo es dividir los documentos respetando, en la medida de lo posible, la estructura normativa:

- títulos;
- capítulos;
- artículos;
- incisos;
- disposiciones;
- anexos.

Estos chunks serán usados luego para generar embeddings e indexarlos en la base vectorial.


## 1. Importación de librerías

In [1]:
from pathlib import Path
from datetime import datetime
import json
import re

try:
    import pandas as pd
    print("pandas importado correctamente.")
except ImportError:
    raise ImportError(
        "No se encontró pandas. Instálalo con: pip install pandas"
    )

print("Librerías importadas correctamente.")


pandas importado correctamente.
Librerías importadas correctamente.


## 2. Definición de rutas del proyecto

El notebook puede ejecutarse desde la raíz del repositorio o desde la carpeta `notebooks/`.


In [2]:
current_dir = Path.cwd()

if current_dir.name == "notebooks":
    ROOT_DIR = current_dir.parent
else:
    ROOT_DIR = current_dir

DATA_DIR = ROOT_DIR / "data"
PROCESSED_DIR = DATA_DIR / "processed"
CHUNKS_DIR = DATA_DIR / "chunks"
METADATA_DIR = DATA_DIR / "metadata"
REPORTS_DIR = ROOT_DIR / "experiments" / "resultados"

CSV_EXTRACTION_PATH = METADATA_DIR / "corpus_normativo_ambiental_con_extraccion.csv"
CSV_BASE_PATH = METADATA_DIR / "corpus_normativo_ambiental.csv"

CHUNKS_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

if CSV_EXTRACTION_PATH.exists():
    CSV_PATH = CSV_EXTRACTION_PATH
else:
    CSV_PATH = CSV_BASE_PATH

print(f"ROOT_DIR: {ROOT_DIR}")
print(f"CSV_PATH: {CSV_PATH}")
print(f"PROCESSED_DIR: {PROCESSED_DIR}")
print(f"CHUNKS_DIR: {CHUNKS_DIR}")


ROOT_DIR: c:\Users\lenovo\Downloads\2026\2026 - A\TABD\Repo_TABD\rag-normativa-ambiental-peruana
CSV_PATH: c:\Users\lenovo\Downloads\2026\2026 - A\TABD\Repo_TABD\rag-normativa-ambiental-peruana\data\metadata\corpus_normativo_ambiental_con_extraccion.csv
PROCESSED_DIR: c:\Users\lenovo\Downloads\2026\2026 - A\TABD\Repo_TABD\rag-normativa-ambiental-peruana\data\processed
CHUNKS_DIR: c:\Users\lenovo\Downloads\2026\2026 - A\TABD\Repo_TABD\rag-normativa-ambiental-peruana\data\chunks


## 3. Carga del corpus

In [3]:
df = pd.read_csv(CSV_PATH)

print(f"Documentos registrados: {len(df)}")
df.head()


Documentos registrados: 30


,id_documento,nombre_archivo_original,archivo_pdf,titulo_documento,tipo_norma,numero_norma,entidad_emisora,fecha_publicacion,tema_principal,subtema,...,url_fuente,texto_extraible,estado_vigencia,prioridad,observaciones,archivo_txt,estado_extraccion,caracteres_extraidos,paginas_pdf,texto_extraible_detectado
0,DOC-001,625.pdf,DOC-001_DS_004_2012_MINAM_IGAC.pdf,Aprueban Disposiciones Complementarias para el...,Decreto Supremo,Decreto Supremo N.º 004-2012-MINAM,MINAM,2012-09-06,Gestión ambiental,Minería y ambiente / IGAC,...,No determinado,Sí,No determinado,Alta,Norma relevante porque regula el IGAC aplicabl...,DOC-001.txt,OK,42050,7,Sí
1,DOC-002,305-2017-MINAM.pdf,DOC-002_RM_305_2017_MINAM_CALIDAD_AIRE_GESTA.pdf,Aprueban los Lineamientos para el Fortalecimie...,Resolución Ministerial,Resolución Ministerial N.º 305-2017-MINAM,MINAM,2017-10-19,Calidad ambiental,ECA / Calidad del aire / GESTA,...,No determinado,Parcial,No determinado,Alta,Norma relevante porque aprueba lineamientos pa...,DOC-002.txt,OK,29068,13,Sí
2,DOC-003,656.pdf,DOC-003_RM_702_2008_MINSA_RESIDUOS_SEGREGADORE...,Aprueban la NTS N.º 073-2008-MINSA/DIGESA-V.01...,Resolución Ministerial,Resolución Ministerial N.º 702-2008/MINSA / NT...,MINSA / DIGESA,2008-10-09,Residuos sólidos,Residuos sólidos / Segregadores / Reaprovecham...,...,http://www.minsa.gob.pe/portal/transparencia/n...,No,No determinado,Media,Documento oficial relevante porque establece u...,DOC-003.txt,PARCIAL,242,11,Parcial
3,DOC-004,1466.pdf,DOC-004_LEY_28611_2005_CONGRESO_LEY_GENERAL_AM...,Ley General del Ambiente,Ley,Ley N.º 28611,Congreso de la República,2005-10-15,Gestión ambiental,SNGA / Gestión ambiental / Política ambiental,...,https://www.leyes.congreso.gob.pe/documentos/l...,Sí,No determinado,Alta,Norma base del marco ambiental peruano. Establ...,DOC-004.txt,OK,122923,46,Sí
4,DOC-005,1599.pdf,DOC-005_RM_554_2012_MINSA_RESIDUOS_ESTABLECIMI...,"Aprueban la NTS N.º 096-MINSA/DIGESA-V.01, Nor...",Resolución Ministerial,Resolución Ministerial N.º 554-2012/MINSA / NT...,MINSA / DIGESA,2012-07-03,Residuos sólidos,Residuos sólidos / Residuos hospitalarios / Ma...,...,http://www.minsa.gob.pe/transparencia/dge_norm...,No,No determinado,Media,Documento oficial relevante porque regula la g...,DOC-005.txt,PARCIAL,1369,60,Parcial


## 4. Parámetros del chunking

Estos valores pueden ajustarse durante la fase de experimentación.

- `MAX_WORDS`: tamaño máximo recomendado por chunk.
- `OVERLAP_WORDS`: solapamiento usado cuando un bloque normativo es demasiado largo.
- `MIN_WORDS`: mínimo de palabras para conservar un chunk.


In [4]:
MAX_WORDS = 450
OVERLAP_WORDS = 80
MIN_WORDS = 25

print(f"MAX_WORDS: {MAX_WORDS}")
print(f"OVERLAP_WORDS: {OVERLAP_WORDS}")
print(f"MIN_WORDS: {MIN_WORDS}")


MAX_WORDS: 450
OVERLAP_WORDS: 80
MIN_WORDS: 25


## 5. Expresiones regulares para detectar estructura normativa

Se usan patrones flexibles para documentos peruanos, considerando formas como:

- `TÍTULO I`
- `CAPÍTULO II`
- `Artículo 1º.-`
- `Artículo 1.-`
- `DISPOSICIONES COMPLEMENTARIAS`
- `ANEXO Nº 01`


In [5]:
PAGE_MARKER_RE = re.compile(r"^=+\s*PÁGINA\s+(\d+)\s*=+$", re.IGNORECASE)

TITLE_RE = re.compile(
    r"^(T[ÍI]TULO\s+(?:PRELIMINAR|[IVXLCDM]+|\d+).*)$",
    re.IGNORECASE
)

CHAPTER_RE = re.compile(
    r"^(CAP[ÍI]TULO\s+(?:[IVXLCDM]+|\d+).*)$",
    re.IGNORECASE
)

ARTICLE_RE = re.compile(
    r"^(Art[íi]culo|Articulo)\s+\d+[A-Za-z0-9]*[°º]?\s*[\.\-–—º]*\s*.*$",
    re.IGNORECASE
)

ANNEX_RE = re.compile(
    r"^(ANEXO\s*(?:N[°º]\s*)?(?:\d+|[IVXLCDM]+)?.*)$",
    re.IGNORECASE
)

DISPOSITION_RE = re.compile(
    r"^((DISPOSICI[ÓO]N|DISPOSICIONES)\s+(COMPLEMENTARIAS|FINALES|TRANSITORIAS|MODIFICATORIAS).*)$",
    re.IGNORECASE
)

LEGAL_HEADER_RE = re.compile(
    r"^(DECRETO SUPREMO|RESOLUCI[ÓO]N MINISTERIAL|LEY|DECRETO LEGISLATIVO|RESOLUCI[ÓO]N DIRECTORAL).*$",
    re.IGNORECASE
)

def normalize_line(line: str) -> str:
    if not isinstance(line, str):
        return ""
    line = line.replace("\t", " ")
    line = re.sub(r" {2,}", " ", line)
    return line.strip()


def normalize_text_for_chunk(text: str) -> str:
    text = re.sub(r"\n{3,}", "\n\n", text)
    text = re.sub(r"[ \t]{2,}", " ", text)
    return text.strip()


def count_words(text: str) -> int:
    return len(re.findall(r"\b\w+\b", text, flags=re.UNICODE))


## 6. Funciones principales de chunking

In [7]:
def split_large_text(text: str, max_words: int = MAX_WORDS, overlap_words: int = OVERLAP_WORDS):
    """
    Divide un texto largo en subchunks con solapamiento.
    Se usa solo cuando un artículo, anexo o bloque excede el tamaño máximo.
    """
    words = text.split()

    if len(words) <= max_words:
        return [text]

    chunks = []
    start = 0

    while start < len(words):
        end = start + max_words
        chunk_words = words[start:end]
        chunks.append(" ".join(chunk_words))

        if end >= len(words):
            break

        start = max(0, end - overlap_words)

    return chunks


def create_chunk_record(
    row,
    text,
    tipo_chunk,
    seccion,
    titulo_seccion,
    capitulo,
    anexo,
    pagina_inicio,
    pagina_fin,
    orden_chunk,
    suborden=1
):
    """
    Construye un registro de chunk con metadatos documentales.
    """
    chunk_id = f"{row['id_documento']}_CHK-{orden_chunk:04d}"

    if suborden > 1:
        chunk_id = f"{chunk_id}_P{suborden:02d}"

    clean_chunk_text = normalize_text_for_chunk(text)

    return {
        "id_chunk": chunk_id,
        "id_documento": row.get("id_documento", "No determinado"),
        "archivo_pdf": row.get("archivo_pdf", "No determinado"),
        "archivo_txt": row.get("archivo_txt", f"{row.get('id_documento', 'DOC')}.txt"),
        "titulo_documento": row.get("titulo_documento", "No determinado"),
        "tipo_norma": row.get("tipo_norma", "No determinado"),
        "numero_norma": row.get("numero_norma", "No determinado"),
        "entidad_emisora": row.get("entidad_emisora", "No determinado"),
        "fecha_publicacion": row.get("fecha_publicacion", "No determinado"),
        "tema_principal": row.get("tema_principal", "No determinado"),
        "subtema": row.get("subtema", "No determinado"),
        "fuente_oficial": row.get("fuente_oficial", "No determinado"),
        "url_fuente": row.get("url_fuente", "No determinado"),
        "estado_vigencia": row.get("estado_vigencia", "No determinado"),
        "prioridad": row.get("prioridad", "No determinado"),
        "tipo_chunk": tipo_chunk,
        "seccion": seccion,
        "titulo_seccion": titulo_seccion,
        "capitulo": capitulo,
        "anexo": anexo,
        "pagina_inicio": pagina_inicio,
        "pagina_fin": pagina_fin,
        "orden_chunk": orden_chunk,
        "suborden": suborden,
        "n_palabras": count_words(clean_chunk_text),
        "n_caracteres": len(clean_chunk_text),
        "texto": clean_chunk_text,
    }


def finalize_block(block, row, chunks, orden_chunk):
    """
    Finaliza un bloque actual y lo agrega a la lista de chunks.
    Si el bloque es muy grande, lo divide en subchunks.
    """
    if block is None:
        return orden_chunk

    text = normalize_text_for_chunk("\n".join(block["lines"]))
    word_count = count_words(text)

    if word_count < MIN_WORDS:
        return orden_chunk

    subtexts = split_large_text(text, MAX_WORDS, OVERLAP_WORDS)

    for suborder, subtext in enumerate(subtexts, start=1):
        chunk_record = create_chunk_record(
            row=row,
            text=subtext,
            tipo_chunk=block["tipo_chunk"],
            seccion=block["seccion"],
            titulo_seccion=block["titulo_seccion"],
            capitulo=block["capitulo"],
            anexo=block["anexo"],
            pagina_inicio=block["pagina_inicio"],
            pagina_fin=block["pagina_fin"],
            orden_chunk=orden_chunk,
            suborden=suborder
        )
        chunks.append(chunk_record)

    return orden_chunk + 1


def structural_chunk_document(text: str, row) -> list:
    """
    Genera chunks estructurales a partir del texto de un documento.
    El algoritmo recorre línea por línea y detecta páginas, títulos, capítulos,
    artículos, disposiciones y anexos.
    """
    chunks = []

    current_page = None
    current_title = "No determinado"
    current_chapter = "No determinado"
    current_annex = "No determinado"

    current_block = None
    orden_chunk = 1

    lines = text.splitlines()

    for raw_line in lines:
        line = normalize_line(raw_line)

        if not line:
            continue

        page_match = PAGE_MARKER_RE.match(line)
        if page_match:
            current_page = int(page_match.group(1))
            if current_block is not None:
                current_block["pagina_fin"] = current_page
            continue

        if TITLE_RE.match(line):
            current_title = line
            continue

        if CHAPTER_RE.match(line):
            current_chapter = line
            continue

        if ANNEX_RE.match(line):
            orden_chunk = finalize_block(current_block, row, chunks, orden_chunk)
            current_annex = line
            current_block = {
                "tipo_chunk": "anexo",
                "seccion": line,
                "titulo_seccion": current_title,
                "capitulo": current_chapter,
                "anexo": current_annex,
                "pagina_inicio": current_page,
                "pagina_fin": current_page,
                "lines": [line],
            }
            continue

        if DISPOSITION_RE.match(line):
            orden_chunk = finalize_block(current_block, row, chunks, orden_chunk)
            current_block = {
                "tipo_chunk": "disposicion",
                "seccion": line,
                "titulo_seccion": current_title,
                "capitulo": current_chapter,
                "anexo": current_annex,
                "pagina_inicio": current_page,
                "pagina_fin": current_page,
                "lines": [line],
            }
            continue

        if ARTICLE_RE.match(line):
            orden_chunk = finalize_block(current_block, row, chunks, orden_chunk)
            current_block = {
                "tipo_chunk": "articulo",
                "seccion": line,
                "titulo_seccion": current_title,
                "capitulo": current_chapter,
                "anexo": current_annex,
                "pagina_inicio": current_page,
                "pagina_fin": current_page,
                "lines": [line],
            }
            continue

        if LEGAL_HEADER_RE.match(line) and current_block is None:
            current_block = {
                "tipo_chunk": "encabezado_normativo",
                "seccion": line,
                "titulo_seccion": current_title,
                "capitulo": current_chapter,
                "anexo": current_annex,
                "pagina_inicio": current_page,
                "pagina_fin": current_page,
                "lines": [line],
            }
            continue

        if current_block is None:
            current_block = {
                "tipo_chunk": "preambulo",
                "seccion": "Preámbulo / considerandos",
                "titulo_seccion": current_title,
                "capitulo": current_chapter,
                "anexo": current_annex,
                "pagina_inicio": current_page,
                "pagina_fin": current_page,
                "lines": [],
            }

        current_block["lines"].append(line)
        current_block["pagina_fin"] = current_page

    orden_chunk = finalize_block(current_block, row, chunks, orden_chunk)

    return chunks


## 7. Prueba de chunking con el primer documento

Antes de procesar todo el corpus, se revisa un documento de ejemplo.


In [8]:
first_row = df.iloc[0].to_dict()

archivo_txt = first_row.get("archivo_txt", f"{first_row['id_documento']}.txt")
txt_path = PROCESSED_DIR / archivo_txt

print(f"Documento de prueba: {first_row['id_documento']}")
print(f"Archivo TXT: {txt_path}")

if not txt_path.exists():
    print(f"No se encontró el archivo de texto: {txt_path}")
else:
    text = txt_path.read_text(encoding="utf-8", errors="ignore")
    sample_chunks = structural_chunk_document(text, first_row)

    print(f"Chunks generados para el documento de prueba: {len(sample_chunks)}")

    if sample_chunks:
        print("\nPrimer chunk generado:")
        print(json.dumps({k: v for k, v in sample_chunks[0].items() if k != "texto"}, ensure_ascii=False, indent=4))
        print("\nTexto del primer chunk:")
        print(sample_chunks[0]["texto"][:1500])


Documento de prueba: DOC-001
Archivo TXT: c:\Users\lenovo\Downloads\2026\2026 - A\TABD\Repo_TABD\rag-normativa-ambiental-peruana\data\processed\DOC-001.txt
Chunks generados para el documento de prueba: 37

Primer chunk generado:
{
    "id_chunk": "DOC-001_CHK-0001",
    "id_documento": "DOC-001",
    "archivo_pdf": "DOC-001_DS_004_2012_MINAM_IGAC.pdf",
    "archivo_txt": "DOC-001.txt",
    "titulo_documento": "Aprueban Disposiciones Complementarias para el Instrumento de Gestión Ambiental Correctivo - IGAC para la Formalización de Actividades de Pequeña Minería y Minería Artesanal en curso",
    "tipo_norma": "Decreto Supremo",
    "numero_norma": "Decreto Supremo N.º 004-2012-MINAM",
    "entidad_emisora": "MINAM",
    "fecha_publicacion": "2012-09-06",
    "tema_principal": "Gestión ambiental",
    "subtema": "Minería y ambiente / IGAC",
    "fuente_oficial": "El Peruano",
    "url_fuente": "No determinado",
    "estado_vigencia": "No determinado",
    "prioridad": "Alta",
    "tipo_

## 8. Generación de chunks para todos los documentos

Se procesan todos los textos extraídos y se genera un conjunto único de chunks.


In [9]:
all_chunks = []
chunking_results = []

for _, row in df.iterrows():
    row_dict = row.to_dict()
    id_documento = str(row_dict["id_documento"])

    archivo_txt = row_dict.get("archivo_txt", None)

    if pd.isna(archivo_txt) or not archivo_txt:
        archivo_txt = f"{id_documento}.txt"

    txt_path = PROCESSED_DIR / str(archivo_txt)

    if not txt_path.exists():
        chunking_results.append({
            "id_documento": id_documento,
            "archivo_txt": str(archivo_txt),
            "estado_chunking": "TXT_NO_ENCONTRADO",
            "n_chunks": 0,
            "n_palabras_total_chunks": 0,
            "observacion": f"No existe {txt_path}",
        })
        continue

    text = txt_path.read_text(encoding="utf-8", errors="ignore")
    chunks = structural_chunk_document(text, row_dict)

    all_chunks.extend(chunks)

    chunking_results.append({
        "id_documento": id_documento,
        "archivo_txt": str(archivo_txt),
        "estado_chunking": "OK" if chunks else "SIN_CHUNKS",
        "n_chunks": len(chunks),
        "n_palabras_total_chunks": sum(chunk["n_palabras"] for chunk in chunks),
        "observacion": "Chunking completado" if chunks else "No se generaron chunks válidos",
    })

chunks_df = pd.DataFrame(all_chunks)
chunking_results_df = pd.DataFrame(chunking_results)

print(f"Total de chunks generados: {len(chunks_df)}")
chunks_df.head()


Total de chunks generados: 1057


,id_chunk,id_documento,archivo_pdf,archivo_txt,titulo_documento,tipo_norma,numero_norma,entidad_emisora,fecha_publicacion,tema_principal,...,titulo_seccion,capitulo,anexo,pagina_inicio,pagina_fin,orden_chunk,suborden,n_palabras,n_caracteres,texto
0,DOC-001_CHK-0001,DOC-001,DOC-001_DS_004_2012_MINAM_IGAC.pdf,DOC-001.txt,Aprueban Disposiciones Complementarias para el...,Decreto Supremo,Decreto Supremo N.º 004-2012-MINAM,MINAM,2012-09-06,Gestión ambiental,...,No determinado,No determinado,No determinado,1,1,1,1,453,2860,"NORMAS LEGALES El Peruano Lima, jueves 6 de se..."
1,DOC-001_CHK-0001_P02,DOC-001,DOC-001_DS_004_2012_MINAM_IGAC.pdf,DOC-001.txt,Aprueban Disposiciones Complementarias para el...,Decreto Supremo,Decreto Supremo N.º 004-2012-MINAM,MINAM,2012-09-06,Gestión ambiental,...,No determinado,No determinado,No determinado,1,1,1,2,295,1878,así como en el proceso al que se reﬁ ere el De...
2,DOC-001_CHK-0002,DOC-001,DOC-001_DS_004_2012_MINAM_IGAC.pdf,DOC-001.txt,Aprueban Disposiciones Complementarias para el...,Decreto Supremo,Decreto Supremo N.º 004-2012-MINAM,MINAM,2012-09-06,Gestión ambiental,...,No determinado,No determinado,No determinado,1,1,2,1,58,402,Artículo 1º.- Del objeto de la norma\nApruébes...
3,DOC-001_CHK-0003,DOC-001,DOC-001_DS_004_2012_MINAM_IGAC.pdf,DOC-001.txt,Aprueban Disposiciones Complementarias para el...,Decreto Supremo,Decreto Supremo N.º 004-2012-MINAM,MINAM,2012-09-06,Gestión ambiental,...,No determinado,No determinado,No determinado,1,1,3,1,25,144,Artículo 2º.- De la vigencia\nEl presente Decr...
4,DOC-001_CHK-0004,DOC-001,DOC-001_DS_004_2012_MINAM_IGAC.pdf,DOC-001.txt,Aprueban Disposiciones Complementarias para el...,Decreto Supremo,Decreto Supremo N.º 004-2012-MINAM,MINAM,2012-09-06,Gestión ambiental,...,No determinado,No determinado,No determinado,1,1,4,1,51,311,Artículo 3º.- Del refrendo\nEl presente Decret...


## 9. Resumen del chunking

In [10]:
print("Resumen por estado de chunking:")
display(chunking_results_df["estado_chunking"].value_counts().reset_index().rename(columns={"index": "estado", "estado_chunking": "cantidad"}))

print("Resumen por tipo de chunk:")
if not chunks_df.empty:
    display(chunks_df["tipo_chunk"].value_counts().reset_index().rename(columns={"index": "tipo_chunk", "tipo_chunk": "cantidad"}))

print("Documentos con más chunks:")
display(chunking_results_df.sort_values("n_chunks", ascending=False).head(10))

print("Documentos con menos chunks:")
display(chunking_results_df.sort_values("n_chunks").head(10))


Resumen por estado de chunking:


,cantidad,count
0,OK,26
1,SIN_CHUNKS,4


Resumen por tipo de chunk:


,cantidad,count
0,articulo,797
1,anexo,111
2,preambulo,89
3,disposicion,55
4,encabezado_normativo,5


Documentos con más chunks:


,id_documento,archivo_txt,estado_chunking,n_chunks,n_palabras_total_chunks,observacion
22,DOC-023,DOC-023.txt,OK,333,44477,Chunking completado
3,DOC-004,DOC-004.txt,OK,158,18429,Chunking completado
9,DOC-010,DOC-010.txt,OK,119,26857,Chunking completado
13,DOC-014,DOC-014.txt,OK,70,28539,Chunking completado
20,DOC-021,DOC-021.txt,OK,38,16852,Chunking completado
0,DOC-001,DOC-001.txt,OK,37,6113,Chunking completado
8,DOC-009,DOC-009.txt,OK,37,13599,Chunking completado
24,DOC-025,DOC-025.txt,OK,29,5757,Chunking completado
16,DOC-017,DOC-017.txt,OK,25,8859,Chunking completado
6,DOC-007,DOC-007.txt,OK,20,3352,Chunking completado


Documentos con menos chunks:


,id_documento,archivo_txt,estado_chunking,n_chunks,n_palabras_total_chunks,observacion
26,DOC-027,DOC-027.txt,SIN_CHUNKS,0,0,No se generaron chunks válidos
2,DOC-003,DOC-003.txt,SIN_CHUNKS,0,0,No se generaron chunks válidos
4,DOC-005,DOC-005.txt,SIN_CHUNKS,0,0,No se generaron chunks válidos
23,DOC-024,DOC-024.txt,SIN_CHUNKS,0,0,No se generaron chunks válidos
29,DOC-030,DOC-030.txt,OK,5,1135,Chunking completado
25,DOC-026,DOC-026.txt,OK,6,2424,Chunking completado
10,DOC-011,DOC-011.txt,OK,7,1207,Chunking completado
14,DOC-015,DOC-015.txt,OK,8,1764,Chunking completado
28,DOC-029,DOC-029.txt,OK,9,3449,Chunking completado
5,DOC-006,DOC-006.txt,OK,11,3263,Chunking completado


## 10. Control de calidad de chunks

Se revisan chunks demasiado pequeños o demasiado grandes.


In [11]:
if chunks_df.empty:
    print("No hay chunks para revisar.")
else:
    small_chunks = chunks_df[chunks_df["n_palabras"] < MIN_WORDS]
    large_chunks = chunks_df[chunks_df["n_palabras"] > MAX_WORDS]

    print(f"Chunks pequeños (< {MIN_WORDS} palabras): {len(small_chunks)}")
    print(f"Chunks grandes (> {MAX_WORDS} palabras): {len(large_chunks)}")

    print("\nDistribución de palabras por chunk:")
    display(chunks_df["n_palabras"].describe())

    if not large_chunks.empty:
        print("\nMuestra de chunks grandes:")
        display(large_chunks[["id_chunk", "id_documento", "tipo_chunk", "seccion", "n_palabras"]].head(10))


Chunks pequeños (< 25 palabras): 0
Chunks grandes (> 450 palabras): 179

Distribución de palabras por chunk:


count    1057.000000
mean      210.133396
std       163.340571
min        25.000000
25%        72.000000
50%       139.000000
75%       400.000000
max       635.000000
Name: n_palabras, dtype: float64


Muestra de chunks grandes:


,id_chunk,id_documento,tipo_chunk,seccion,n_palabras
0,DOC-001_CHK-0001,DOC-001,preambulo,Preámbulo / considerandos,453
8,DOC-001_CHK-0008,DOC-001,articulo,Artículo 3º.- Deﬁ niciones,451
37,DOC-002_CHK-0001,DOC-002,preambulo,Preámbulo / considerandos,524
38,DOC-002_CHK-0001_P02,DOC-002,preambulo,Preámbulo / considerandos,480
39,DOC-002_CHK-0001_P03,DOC-002,preambulo,Preámbulo / considerandos,520
40,DOC-002_CHK-0001_P04,DOC-002,preambulo,Preámbulo / considerandos,531
41,DOC-002_CHK-0001_P05,DOC-002,preambulo,Preámbulo / considerandos,530
42,DOC-002_CHK-0001_P06,DOC-002,preambulo,Preámbulo / considerandos,543
43,DOC-002_CHK-0001_P07,DOC-002,preambulo,Preámbulo / considerandos,544
44,DOC-002_CHK-0001_P08,DOC-002,preambulo,Preámbulo / considerandos,566


## 11. Muestra aleatoria de chunks

In [12]:
if not chunks_df.empty:
    sample_display = chunks_df.sample(min(5, len(chunks_df)), random_state=42)

    for _, chunk in sample_display.iterrows():
        print("=" * 100)
        print(f"ID chunk: {chunk['id_chunk']}")
        print(f"Documento: {chunk['id_documento']} | {chunk['titulo_documento']}")
        print(f"Tipo chunk: {chunk['tipo_chunk']}")
        print(f"Sección: {chunk['seccion']}")
        print(f"Páginas: {chunk['pagina_inicio']} - {chunk['pagina_fin']}")
        print(f"Palabras: {chunk['n_palabras']}")
        print("-" * 100)
        print(chunk["texto"][:1000])
        print()


ID chunk: DOC-009_CHK-0006
Documento: DOC-009 | Decreto Supremo que modifica el Reglamento de Protección Ambiental para el Sector Transportes, aprobado por Decreto Supremo N.º 004-2017-MTC
Tipo chunk: articulo
Sección: Artículo 3.- Modificación del Reglamento de
Páginas: 2 - 2
Palabras: 46
----------------------------------------------------------------------------------------------------
Artículo 3.- Modificación del Reglamento de
Protección Ambiental para el Sector Transportes,
aprobado por Decreto Supremo Nº 004-2017-MTC
Se modifican los numerales 1, 2, 6 y 7 del artículo 4,
así como los artículos 10, 11, 11-A, 12, 24, 46, 55 y el

ID chunk: DOC-023_CHK-0166
Documento: DOC-023 | Ley General del Ambiente, Ley Marco del Sistema Nacional de Gestión Ambiental, Reglamento de la Ley Marco del Sistema Nacional de Gestión Ambiental y Ley de creación, organización y funciones del Ministerio del Ambiente
Tipo chunk: articulo
Sección: Artículo 6.- De los Instrumentos de Gestión y Planificación

## 12. Guardado de chunks y reportes

Se guardan los chunks en dos formatos:

- `JSONL`: recomendado para procesamiento posterior.
- `CSV`: útil para revisión en Excel o Google Sheets.


In [13]:
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

chunks_jsonl_path = CHUNKS_DIR / "chunks_normativa_v1.jsonl"
chunks_csv_path = CHUNKS_DIR / "chunks_normativa_v1.csv"
chunking_report_path = REPORTS_DIR / f"reporte_chunking_{timestamp}.csv"
chunking_summary_path = REPORTS_DIR / f"resumen_chunking_{timestamp}.json"

if not chunks_df.empty:
    with open(chunks_jsonl_path, "w", encoding="utf-8") as f:
        for chunk in all_chunks:
            f.write(json.dumps(chunk, ensure_ascii=False) + "\n")

    chunks_df.to_csv(chunks_csv_path, index=False, encoding="utf-8-sig")

chunking_results_df.to_csv(chunking_report_path, index=False, encoding="utf-8-sig")

summary = {
    "fecha_chunking": datetime.now().isoformat(timespec="seconds"),
    "total_documentos": int(len(df)),
    "total_chunks": int(len(chunks_df)),
    "max_words": MAX_WORDS,
    "overlap_words": OVERLAP_WORDS,
    "min_words": MIN_WORDS,
    "documentos_ok": int((chunking_results_df["estado_chunking"] == "OK").sum()),
    "documentos_sin_chunks": int((chunking_results_df["estado_chunking"] == "SIN_CHUNKS").sum()),
    "documentos_txt_no_encontrado": int((chunking_results_df["estado_chunking"] == "TXT_NO_ENCONTRADO").sum()),
}

if not chunks_df.empty:
    summary["promedio_palabras_por_chunk"] = float(chunks_df["n_palabras"].mean())
    summary["min_palabras_por_chunk"] = int(chunks_df["n_palabras"].min())
    summary["max_palabras_por_chunk"] = int(chunks_df["n_palabras"].max())
else:
    summary["promedio_palabras_por_chunk"] = 0
    summary["min_palabras_por_chunk"] = 0
    summary["max_palabras_por_chunk"] = 0

with open(chunking_summary_path, "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=4)

print(f"Chunks JSONL guardados en: {chunks_jsonl_path}")
print(f"Chunks CSV guardados en: {chunks_csv_path}")
print(f"Reporte de chunking guardado en: {chunking_report_path}")
print(f"Resumen JSON guardado en: {chunking_summary_path}")


Chunks JSONL guardados en: c:\Users\lenovo\Downloads\2026\2026 - A\TABD\Repo_TABD\rag-normativa-ambiental-peruana\data\chunks\chunks_normativa_v1.jsonl
Chunks CSV guardados en: c:\Users\lenovo\Downloads\2026\2026 - A\TABD\Repo_TABD\rag-normativa-ambiental-peruana\data\chunks\chunks_normativa_v1.csv
Reporte de chunking guardado en: c:\Users\lenovo\Downloads\2026\2026 - A\TABD\Repo_TABD\rag-normativa-ambiental-peruana\experiments\resultados\reporte_chunking_20260519_213430.csv
Resumen JSON guardado en: c:\Users\lenovo\Downloads\2026\2026 - A\TABD\Repo_TABD\rag-normativa-ambiental-peruana\experiments\resultados\resumen_chunking_20260519_213430.json


## 13. Resultado final

Si el chunking se generó correctamente, el corpus queda listo para la siguiente fase:

**Notebook 04 - Embeddings e indexación vectorial**.


In [14]:
if chunks_df.empty:
    print("No se generaron chunks. Revisa los textos extraídos o las reglas de segmentación.")
else:
    print("Chunking completado correctamente.")
    print(f"Total de chunks generados: {len(chunks_df)}")
    print("Siguiente paso: generar embeddings e indexar en la base vectorial.")

display(chunking_results_df)


Chunking completado correctamente.
Total de chunks generados: 1057
Siguiente paso: generar embeddings e indexar en la base vectorial.


,id_documento,archivo_txt,estado_chunking,n_chunks,n_palabras_total_chunks,observacion
0,DOC-001,DOC-001.txt,OK,37,6113,Chunking completado
1,DOC-002,DOC-002.txt,OK,12,6113,Chunking completado
2,DOC-003,DOC-003.txt,SIN_CHUNKS,0,0,No se generaron chunks válidos
3,DOC-004,DOC-004.txt,OK,158,18429,Chunking completado
4,DOC-005,DOC-005.txt,SIN_CHUNKS,0,0,No se generaron chunks válidos
5,DOC-006,DOC-006.txt,OK,11,3263,Chunking completado
6,DOC-007,DOC-007.txt,OK,20,3352,Chunking completado
7,DOC-008,DOC-008.txt,OK,15,2405,Chunking completado
8,DOC-009,DOC-009.txt,OK,37,13599,Chunking completado
9,DOC-010,DOC-010.txt,OK,119,26857,Chunking completado
